In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Casts inválidos viram NULL em vez de derrubar o pipeline (exigência do enunciado)
try:
    spark.conf.set("spark.sql.ansi.enabled", "false")
except Exception as e:
    print("Aviso: não foi possível desligar ANSI:", e)

def ultimo_por(df, chave):
    """Deduplicação: mantém só o registro mais recente (ingestion_datetime) por chave.
    Desempate determinístico via hash da linha, pois as cargas ocorreram no mesmo instante."""
    cols_dado = [c for c in df.columns if c != "ingestion_datetime"]
    w = Window.partitionBy(*chave).orderBy(F.col("ingestion_datetime").desc(),
                                           F.hash(*cols_dado).desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def salvar(df, nome):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(f"silver.{nome}"))
    print(nome, "->", spark.table(f"silver.{nome}").count(), "linhas")

In [0]:
b = spark.table("bronze.tb_cotacao_dolar")
b = (b.withColumn("ts", F.expr("try_cast(dataHoraCotacao as timestamp)"))
      .withColumn("data_cotacao", F.to_date("ts"))
      .filter("ts is not null"))

# A API pode devolver mais de um boletim por dia: fica o último do dia
w = Window.partitionBy("data_cotacao").orderBy(F.col("ts").desc(), F.col("ingestion_datetime").desc())
diaria = (b.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
           .select("data_cotacao", F.col("cotacaoCompra").alias("cotacao_compra")))

# Calendário contínuo (inclui fins de semana e feriados)
cal = (diaria.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim"))
       .select(F.explode(F.sequence("ini", "fim", F.expr("interval 1 day"))).alias("data_cotacao")))

# Forward Fill: dia sem cotação herda a do último dia útil disponível
wff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
cot = (cal.join(diaria, "data_cotacao", "left")
          .withColumn("cotacao_compra",
                      F.last("cotacao_compra", ignorenulls=True).over(wff).cast("decimal(10,4)")))
salvar(cot, "tb_cotacao_dolar")
display(cot.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tb_cotacao_dolar -> 7 linhas


data_cotacao,cotacao_compra
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692


In [0]:
info = spark.table("bronze.tb_movies_info").withColumn("id", F.trim("id")).filter("id is not null and id <> ''")
info = ultimo_por(info, ["id"])   # unicidade por filme, versão mais recente

# Status: normaliza (minúsculas, sem hífen/espaço/ruído) ANTES de traduzir.
# "Post-Production", "post production" e "POST PRODUCTION" viram todos "postproduction".
# Valores corrompidos (ex.: datas caídas na coluna por column shift) caem em "Não Informado".
status_norm = F.regexp_replace(F.lower(F.coalesce(F.col("status"), F.lit(""))), r"[^a-z]", "")
status_pt = (F.when(status_norm == "released", "Lançado")
              .when(status_norm == "postproduction", "Pós-Produção")
              .when(status_norm == "inproduction", "Em Produção")
              .when(status_norm == "planned", "Planejado")
              .when(status_norm == "rumored", "Rumores")
              .when(status_norm.isin("canceled", "cancelled"), "Cancelado")
              .otherwise("Não Informado"))

# Data multi-formato. Padrões observados na origem:
#   yyyy-MM-dd (ISO) | dd/MM/yyyy (barra = dia primeiro) | MM-dd-yyyy (hífen = mês primeiro,
#   confirmado por "01-17-2018", em que 17 não pode ser mês).
# Por isso MM-dd-yyyy vem ANTES de dd-MM-yyyy. Só vira NULL se nenhum formato funcionar.
formatos = ["yyyy-MM-dd", "dd/MM/yyyy", "MM-dd-yyyy", "yyyy/MM/dd", "yyyy-M-d", "d/M/yyyy",
            "MM/dd/yyyy", "yyyyMMdd", "yyyy-MM-dd HH:mm:ss", "dd-MM-yyyy", "d MMM yyyy", "MMMM d, yyyy"]
data = F.coalesce(*[F.expr(f"try_to_date(trim(release_date), '{f}')") for f in formatos])

info = info.withColumn("data_lancamento", data)

silver_info = info.select(
    F.col("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    "data_lancamento",
    F.expr("try_cast(try_cast(runtime as double) as int)").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    status_pt.alias("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    F.year("data_lancamento").alias("ano_lancamento"))
salvar(silver_info, "tb_info_filmes")
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count())

tb_info_filmes -> 97611 linhas


status_filme,count
Lançado,96261
Em Produção,604
Pós-Produção,697
Planejado,47
Não Informado,2


In [0]:
fin = spark.table("bronze.tb_movies_financials").withColumn("id", F.trim("id")).filter("id is not null and id <> ''")
fin = ultimo_por(fin, ["id"])

def limpa_moeda(c):
    """Higieniza texto monetário -> DECIMAL(18,2). Zero, negativo e 'ausência' viram NULL."""
    s = F.trim(F.col(c).cast("string"))
    ausente = s.isNull() | F.lower(s).isin("unknown", "não informado", "nao informado",
                                           "nan", "null", "none", "n/a", "na", "")
    negativo = s.rlike(r"^[^0-9]*-")                      # sinal de menos antes dos dígitos
    so = F.regexp_replace(s, r"[^0-9.,]", "")             # remove $, R$, espaços e texto
    plano = s.rlike(r"^\d+(\.\d+)?$")                     # número simples: 3500000 ou 3500000.0
    br = so.rlike(r"^\d{1,3}(\.\d{3})+(,\d+)?$")          # 1.234.567,89
    us = so.rlike(r"^\d{1,3}(,\d{3})+(\.\d+)?$")          # 1,234,567.89
    dec_virg = so.rlike(r"^\d+,\d{1,2}$")                 # 1234,5
    num = (F.when(plano, s)
            .when(br, F.regexp_replace(F.regexp_replace(so, r"\.", ""), ",", "."))
            .when(us, F.regexp_replace(so, ",", ""))
            .when(dec_virg, F.regexp_replace(so, ",", "."))
            .otherwise(F.regexp_replace(so, r"[.,]", "")))
    valido = ~ausente & ~negativo & (so != "") & (F.length(num) <= 16)
    val = F.when(valido, num).cast("decimal(18,2)")
    return F.when(val > 0, val)                           # zerado ou negativo = ausente

# Taxa de conversão: cotação mais recente da série contínua (decisão documentada no README)
taxa = float(spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()["cotacao_compra"])
print("Taxa USD->BRL usada:", taxa)

f = (fin.select(F.col("id").alias("id_filme"),
                limpa_moeda("budget").alias("orcamento_usd"),
                limpa_moeda("revenue").alias("receita_usd"))
     .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(taxa), 2).cast("decimal(18,2)"))
     .withColumn("receita_brl",   F.round(F.col("receita_usd")   * F.lit(taxa), 2).cast("decimal(18,2)"))
     # NULL propaga: sem orçamento ou sem receita, não há lucro (e nada quebra)
     .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
     .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
     # Margem % = lucro / receita * 100; protegida contra divisão por zero/NULL
     .withColumn("margem_lucro_pct",
                 F.when(F.col("receita_usd") > 0,
                        F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("decimal(18,2)")))
salvar(f, "tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").limit(10))

Taxa USD->BRL usada: 4.9692
tb_financeiro_filmes -> 99006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000054,null,null,null,null,null,null,null
1000058,4700000.00,null,23355240.00,null,null,null,null
1000059,null,null,null,null,null,null,null
1000073,6000000.00,null,29815200.00,null,null,null,null


In [0]:
met = spark.table("bronze.tb_movies_metrics").withColumn("id", F.trim("id")).filter("id is not null and id <> ''")
met = ultimo_por(met, ["id"])

def num_seguro(c, inteiro=False):
    """Conversão segura: só converte se o texto parecer número (rejeita textos do column shift)."""
    s = F.trim(F.col(c).cast("string"))
    ok = s.rlike(r"^-?[0-9][0-9.,]*$|^-?[.,][0-9]+$")
    if inteiro:
        limpo = F.regexp_replace(s, ",", "")
    else:
        # vírgula -> ponto; se sobrar mais de um ponto, mantém só o último (decimal)
        limpo = F.regexp_replace(F.regexp_replace(s, ",", "."), r"\.(?=.*\.)", "")
    return F.when(ok, limpo).cast("double")

def nota_valida(c):
    """Escala de negócio 0..10. Fora disso (ex.: 85 por erro de escala) = NULL."""
    v = num_seguro(c)
    return F.when((v >= 0) & (v <= 10), v)

def contagem(c):
    """Contagem de votos: negativos = NULL."""
    v = num_seguro(c, inteiro=True)
    return F.when(v >= 0, F.round(v).cast("int"))

silver_met = met.select(
    F.col("id").alias("id_filme"),
    F.when(num_seguro("popularity") >= 0, num_seguro("popularity")).alias("popularidade"),
    nota_valida("vote_average").alias("nota_media_tmdb"),
    contagem("vote_count").alias("qtd_votos_tmdb"),
    nota_valida("averageRating").alias("nota_media_imdb"),
    contagem("numVotes").alias("qtd_votos_imdb"))
salvar(silver_met, "tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").limit(10))

tb_metricas_engajamento -> 95115 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236
1000081,102.802,5.142,134,5.1,1841
1000092,4.797,8.5,1,6.8,130
1000094,2.645,6.9,28,6.5,null
1000096,0.71,10.0,1,6.3,695
1000099,0.6,0.0,0,8.6,33
1000108,3.231,0.0,0,6.5,157
1000110,1.38,0.0,0,6.9,263


In [0]:
rev = spark.table("bronze.tb_movies_reviews")

# 1) Duplicatas integrais: mesma combinação filme+usuário+nota+comentário (sem considerar ingestion_datetime)
base = (rev.select(F.trim("id").alias("id_filme"),
                   F.trim("nome").alias("nome_usuario"),
                   F.trim(F.col("nota").cast("string")).alias("nota_raw"),
                   F.trim("comentario").alias("comentario_raw"))
           .dropDuplicates())

# 2) Nota na escala 0..10 (fora disso = NULL); comentário vazio/espaços = "Sem comentário"
nota = F.expr("try_cast(nota_raw as double)")
silver_rev = base.select(
    "id_filme", "nome_usuario",
    F.when((nota >= 0) & (nota <= 10), nota).alias("nota_usuario"),
    F.when(F.col("comentario_raw").isNull() | (F.col("comentario_raw") == ""), "Sem comentário")
     .otherwise(F.col("comentario_raw")).alias("comentario_usuario"))
salvar(silver_rev, "tb_avaliacoes_usuarios")

tb_avaliacoes_usuarios -> 32412 linhas


In [0]:
cred = (spark.table("bronze.tb_credits_and_tags")
        .withColumn("id", F.trim("id")).filter("id is not null and id <> ''"))
cred = ultimo_por(cred, ["id"]).withColumn("id_filme", F.col("id"))

def explode_lista(df, coluna):
    """Unifica separadores (; | ,) em vírgula, remove colchetes de listas vazias ('[]'),
    explode e descarta vazios/N/A. Apóstrofos internos (O'Neil) são preservados."""
    s = F.regexp_replace(F.col(coluna).cast("string"), r"[;|]", ",")
    s = F.regexp_replace(s, r"[\[\]]", "")
    return (df.select("id_filme", F.explode(F.split(s, ",")).alias("valor"))
              .withColumn("valor", F.regexp_replace(F.trim("valor"), r"^['\"]+|['\"]+$", ""))
              .withColumn("valor", F.trim("valor"))
              .filter(~F.upper(F.col("valor")).isin("", "N/A", "NA", "NAN", "NULL", "NONE", "UNKNOWN")))

# Regra de negócio: manter só valores que pertencem ao domínio de gêneros (catálogo TMDB).
GENEROS_VALIDOS = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
                   "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
                   "Science Fiction", "Tv Movie", "Thriller", "War", "Western"]

g = (explode_lista(cred, "genres")
     .withColumn("genero", F.initcap("valor"))
     .filter(F.col("genero").isin(GENEROS_VALIDOS))
     .select("id_filme", "genero").distinct())
salvar(g, "tb_generos")
display(spark.table("silver.tb_generos").groupBy("genero").count().orderBy(F.desc("count")))

tb_generos -> 141938 linhas


genero,count
Drama,32615
Documentary,19234
Comedy,18826
Thriller,10400
Horror,9833
Romance,7717
Action,6114
Crime,4786
Animation,4524
Tv Movie,4115


In [0]:
mapa = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

partes = []
for col, tipo in mapa.items():
    p = (explode_lista(cred, col)
          .filter(~F.col("valor").rlike(r"^[\d\s.\-]+$")       # só números/pontuação
                 & ~F.col("valor").rlike(r"[\\\"]")            # sobras de aspas/escape
                 & (F.length("valor") >= 2) & (F.length("valor") <= 80))
         .select("id_filme",
                 F.initcap("valor").alias("nome_entidade"),     # padroniza capitalização
                 F.lit(tipo).alias("tipo_entidade")))
    partes.append(p)

pe = partes[0]
for p in partes[1:]:
    pe = pe.unionByName(p)
salvar(pe.distinct(), "tb_pessoas_empresas")      # distinct = elimina duplicatas
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())

tb_pessoas_empresas -> 890302 linhas


tipo_entidade,count
Ator,542387
Diretor,101509
Roteirista,125957
Produtora,120449


In [0]:
for t in ["tb_info_filmes","tb_financeiro_filmes","tb_metricas_engajamento","tb_avaliacoes_usuarios",
          "tb_generos","tb_pessoas_empresas","tb_cotacao_dolar"]:
    print(t, spark.table(f"silver.{t}").count())

# Unicidade por filme (os 3 devem retornar 0)
for t in ["tb_info_filmes","tb_financeiro_filmes","tb_metricas_engajamento"]:
    d = spark.sql(f"SELECT COUNT(*) c FROM (SELECT id_filme FROM silver.{t} GROUP BY id_filme HAVING COUNT(*)>1)").first()["c"]
    print("ids duplicados em", t, ":", d)

display(spark.table("silver.tb_info_filmes").limit(5))
display(spark.table("silver.tb_financeiro_filmes").limit(5))
display(spark.table("silver.tb_metricas_engajamento").limit(5))

tb_info_filmes 97611
tb_financeiro_filmes 99006
tb_metricas_engajamento 95115
tb_avaliacoes_usuarios 32412
tb_generos 141938
tb_pessoas_empresas 890302
tb_cotacao_dolar 7
ids duplicados em tb_info_filmes : 0
ids duplicados em tb_financeiro_filmes : 0
ids duplicados em tb_metricas_engajamento : 0


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On va manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236
1000081,102.802,5.142,134,5.1,1841
1000092,4.797,8.5,1,6.8,130
